<a href="https://colab.research.google.com/github/Aireenelz/WIE3007-DMW-Group6/blob/xgboost-modelling-joan/xgboost_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### 1. Setup and Data Loading
##### - load the synthetic_financial_data into google colab
##### - initial observation of the dataset

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('synthetic_financial_data.csv')

# Quick look
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (1000, 22)


,CustomerID,Age,Education,EmploymentType,EmploymentSector,EmploymentLengthYears,MonthlyIncome,MaritalStatus,Dependents,PropertyOwnership,...,MonthlyDebt,YearsWithBank,HasSavingsAccount,HasCheckingAccount,LoanPurpose,LoanAmount,LoanTermMonths,InterestRate,LoanDefault,LoanPurposeDescription
0,1,59,High School,Self-Employed,Technology,11,9067.159786,Single,2,Own with Mortgage,...,1583.065500,7,1,0,Personal,20789.005502,36,3.828919,0,Need funds for the next three years.
1,2,51,Below High School,Full-Time,Government,24,5068.058742,Married,1,Own with Mortgage,...,1332.052087,7,1,1,Personal,27461.583712,48,8.855769,1,I need a personal loan for this to be done.
2,3,24,Bachelor,Unemployed,Manufacturing,0,2557.541824,Single,0,Living with Parents,...,911.536097,2,1,1,Business,68862.744645,240,14.748993,1,Seeking financing to start my own company.
3,4,25,Bachelor,Full-Time,Finance,0,7535.097299,Single,0,Living with Parents,...,514.141526,6,1,0,Wedding,8841.450954,48,4.825886,0,I need a personal loan for me.
4,5,25,Master,Part-Time,Government,5,4798.465251,Married,2,Rent,...,2386.488285,3,0,1,Education,17787.927594,120,18.384157,1,"Applying for student financing for tuition, fe..."


#### 2. Small Language Models (SLM) Feature Extraction
##### - use DistillBERT model for sentiment analysis of column 'LoanPurposeDescription'
##### - create a new column 'Sentiment_Feature' that categorize the 'LoanPurposeDescription' into 'POSITIVE', 'NEGATIVE, and 'NEUTRAL'


In [ ]:
# import necessary library
from transformers import pipeline

# Load a lightweight sentiment model (distillbert model) for sentiment analysis
sentiment_task = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
# sentiment-analysis: tell the model we want sentiment analysis so it looks for positive or negative emotions in text

# get_sentiment: function to process text
def get_sentiment(text):
    # if a row has no text, just mark it as "NEUTRAL"
    if pd.isna(text) or text == "": return "NEUTRAL"
    # [:512]: the limit because BER-style model can only read 512 characters at a time
    # ['label']: The model returns a lot of data, but we only want the 'label' (e.g., "POSITIVE" or "NEGATIVE").
    return sentiment_task(text[:512])[0]['label']

# create a new column 'Sentiment_Feature'
# it takes the text in LoanPurposeDescription and converts it into a category
df['Original_Sentiment_Feature'] = df['LoanPurposeDescription'].apply(get_sentiment)
print(df['Original_Sentiment_Feature'].value_counts())
df[['LoanPurposeDescription', 'Original_Sentiment_Feature']].head(10)

Device set to use cuda:0


Original_Sentiment_Feature
NEGATIVE    872
NEUTRAL      66
POSITIVE     62
Name: count, dtype: int64


,LoanPurposeDescription,Original_Sentiment_Feature
0,Need funds for the next three years.,NEGATIVE
1,I need a personal loan for this to be done.,NEGATIVE
2,Seeking financing to start my own company.,NEGATIVE
3,I need a personal loan for me.,NEGATIVE
4,"Applying for student financing for tuition, fe...",NEGATIVE
5,Applying for financing to cover the cost of th...,NEGATIVE
6,I need a personal loan for me.,NEGATIVE
7,Applying for financing to cover the costs of t...,NEGATIVE
8,Applying for financing to cover the costs of t...,NEGATIVE
9,I need a personal loan for me to take care of ...,NEGATIVE


##### It is very common for financial datasets to lean "negative" when using general sentiment models because words like "Debt," "Emergency," or "Loan" are often flagged as negative.

To get a better mix of Positive, Negative, and Neutral, a Keyword-Augmented Logic is used.

The Strategy:
- Neutral: Keep NaN as Neutral.

- Positive Keywords: If words like "Growth," "Expansion," "Investment," or "Success" appear, we nudge the result to Positive.

- Negative Keywords: If words like "Struggling," "Debt," "Emergency," or "Late" appear, we nudge it to Negative.

- AI Fallback: If no keywords are found, we let the DistilBERT model decide.

In [ ]:
# improve the sentiment analysis

# import necessary library
from transformers import pipeline

# Load a lightweight sentiment model (distillbert model) for sentiment analysis
sentiment_task = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
# sentiment-analysis: tell the model we want sentiment analysis so it looks for positive or negative emotions in text

# get_sentiment: function to process text
def get_improved_sentiment(text):
    # # if a row has no text, just mark it as "NEUTRAL"
    if pd.isna(text) or text == "" or text == "None":
        return "NEUTRAL"

    text_lower = str(text).lower()

    # define Financial Keywords
    positive_words = ['business', 'expand', 'opportunity','student', 'starting', 'future','career','education','new']
    negative_words = ['struggling', 'debt', 'emergency', 'overdue', 'bills', 'medical', 'crisis', 'urgent']

    # keyword
    if any(word in text_lower for word in positive_words):
        return "POSITIVE"
    if any(word in text_lower for word in negative_words):
        return "NEGATIVE"

    # AI Fallback (If no keywords found, let the model decide)
    try:
        result = sentiment_task(text[:512])[0]['label']
        return result
    except:
        return "NEUTRAL"

# Apply the function
df['Sentiment_Feature'] = df['LoanPurposeDescription'].apply(get_improved_sentiment)

print(df['Sentiment_Feature'].value_counts())
df[['LoanPurposeDescription', 'Sentiment_Feature']].head(10)

Device set to use cuda:0


Sentiment_Feature
NEGATIVE    634
POSITIVE    300
NEUTRAL      66
Name: count, dtype: int64


,LoanPurposeDescription,Sentiment_Feature
0,Need funds for the next three years.,NEGATIVE
1,I need a personal loan for this to be done.,NEGATIVE
2,Seeking financing to start my own company.,NEGATIVE
3,I need a personal loan for me.,NEGATIVE
4,"Applying for student financing for tuition, fe...",POSITIVE
5,Applying for financing to cover the cost of th...,NEGATIVE
6,I need a personal loan for me.,NEGATIVE
7,Applying for financing to cover the costs of t...,NEGATIVE
8,Applying for financing to cover the costs of t...,NEGATIVE
9,I need a personal loan for me to take care of ...,NEGATIVE


#### 3. AI-Driven Risk Categorization using Zero-Shot Classification Pipeline (BART-Large)
##### - Zero-Shot Classification: this model compares the "meaning" of the loan description to the "meaning" of your labels.
##### - The Keyword Layer: add a manual check for words like "crisis" or "expansion." This acts as a "Financial Expert" rule-base, which makes the model more reliable for banking data.

In [ ]:
# load the Zero-Shot Classification Pipeline (BART-Large)
risk_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

def get_risk_category(text):
    # Handle missing data
    if pd.isna(text) or text == "" or text == "None":
        return "UNKNOWN"

    text_lower = str(text).lower()

    # Financial Domain Keywords (Fast Logic)
    high_risk_words = ['emergency', 'medical', 'struggling', 'bills', 'overdue', 'debt', 'urgent', 'crisis']
    low_risk_words = ['expansion', 'investment', 'growth', 'business', 'mortgage', 'assets', 'startup']

    if any(word in text_lower for word in high_risk_words):
        return "HIGH RISK"
    if any(word in text_lower for word in low_risk_words):
        return "LOW RISK"

    # AI Fallback (For descriptions that are hard to judge)
    try:
        candidate_labels = ["high risk", "medium risk", "low risk"]
        result = risk_classifier(text[:512], candidate_labels)
        return result['labels'][0].upper() # Returns the label with the highest score
    except:
        return "MEDIUM RISK"

# Apply the risk categorization (Applying to all, but AI part only triggers if keywords fail)
df['AI_Risk_Category'] = df['LoanPurposeDescription'].apply(get_risk_category)

# Check the results
print(df['AI_Risk_Category'].value_counts())
df[['LoanPurposeDescription', 'AI_Risk_Category']].head(10)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


AI_Risk_Category
MEDIUM RISK    439
HIGH RISK      353
LOW RISK       142
UNKNOWN         66
Name: count, dtype: int64


,LoanPurposeDescription,AI_Risk_Category
0,Need funds for the next three years.,MEDIUM RISK
1,I need a personal loan for this to be done.,HIGH RISK
2,Seeking financing to start my own company.,MEDIUM RISK
3,I need a personal loan for me.,MEDIUM RISK
4,"Applying for student financing for tuition, fe...",MEDIUM RISK
5,Applying for financing to cover the cost of th...,HIGH RISK
6,I need a personal loan for me.,MEDIUM RISK
7,Applying for financing to cover the costs of t...,HIGH RISK
8,Applying for financing to cover the costs of t...,HIGH RISK
9,I need a personal loan for me to take care of ...,HIGH RISK


#### 4. Preprocessing
##### After extracting AI features (Sentiment and Risk), we need to prepare the entire dataset for the XGBoost model.

#####- Label Encoding: XGBoost is a mathematical algorithm; it only understands numbers. We need to encode the data.
##### - Train-Test Split: split the data into a Training Set (to teach the model) and a Testing Set (to see if it actually learned).

In [ ]:
df_dropped = df.drop(columns=['Original_Sentiment_Feature'])

In [ ]:
df_dropped.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              1000 non-null   int64  
 1   Age                     1000 non-null   int64  
 2   Education               1000 non-null   object 
 3   EmploymentType          1000 non-null   object 
 4   EmploymentSector        1000 non-null   object 
 5   EmploymentLengthYears   1000 non-null   int64  
 6   MonthlyIncome           1000 non-null   float64
 7   MaritalStatus           1000 non-null   object 
 8   Dependents              1000 non-null   int64  
 9   PropertyOwnership       1000 non-null   object 
 10  CreditScore             1000 non-null   int64  
 11  ExistingLoans           1000 non-null   int64  
 12  MonthlyDebt             1000 non-null   float64
 13  YearsWithBank           1000 non-null   int64  
 14  HasSavingsAccount       1000 non-null   i

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Identify your columns
# Categorical columns need to be turned into numbers
categorical_cols = [
    'Education', 'EmploymentType', 'EmploymentSector', 'MaritalStatus',
    'PropertyOwnership','LoanPurpose', 'LoanPurposeDescription', 'Sentiment_Feature', 'AI_Risk_Category'
]

# Numerical columns stay as they are
numerical_cols = [
    'CustomerID','Age', 'EmploymentLengthYears', 'MonthlyIncome', 'Dependents',
    'CreditScore', 'ExistingLoans', 'MonthlyDebt', 'YearsWithBank', 'HasSavingsAccount',
    'HasCheckingAccount','LoanAmount', 'LoanTermMonths', 'InterestRate'
]

# Label Encoding
# This turns "High Risk" -> 2, "Medium Risk" -> 1, "Low Risk" -> 0
le = LabelEncoder()
for col in categorical_cols:
    df_dropped[col] = le.fit_transform(df_dropped[col].astype(str))

# Define X (Features) and y (Target)
X = df_dropped[numerical_cols + categorical_cols]
y = df_dropped['LoanDefault'] # This is what we want to predict

# Split the Data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")
X_train.head()

Training rows: 800
Testing rows: 200


,CustomerID,Age,EmploymentLengthYears,MonthlyIncome,Dependents,CreditScore,ExistingLoans,MonthlyDebt,YearsWithBank,HasSavingsAccount,...,InterestRate,Education,EmploymentType,EmploymentSector,MaritalStatus,PropertyOwnership,LoanPurpose,LoanPurposeDescription,Sentiment_Feature,AI_Risk_Category
29,30,54,11,6524.890658,0,672,1,2141.432699,12,1,...,8.148805,3,1,3,0,2,3,290,2,2
535,536,21,0,1604.272714,0,553,2,551.021990,2,1,...,13.011506,0,4,5,2,0,2,76,0,0
695,696,66,30,7677.177003,0,752,1,455.796929,17,1,...,3.656724,1,1,7,0,2,9,263,0,2
557,558,19,0,722.143859,0,584,2,347.105904,1,1,...,14.422989,2,2,2,2,0,3,392,2,1
836,837,67,20,9672.124896,3,758,2,971.675691,8,1,...,5.343914,1,1,5,1,1,0,555,1,3


#### 5. Export the preprocessed data

In [ ]:
# Save the preprocessed data to a CSV
df_dropped.to_csv('preprocessed_financial_data.csv', index=False)

print("File saved.")

File saved.
